# 03 — Trích Xuất Keypoints 3D & Phân Tích Dữ Liệu (EDA)

Notebook này thực hiện giai đoạn cuối cùng của pipeline xử lý dữ liệu VSL-400:

1. **Trích xuất 76 keypoints 3D** (34 body + 42 hands) từ video đã tiền xử lý
   bằng **MediaPipe Holistic**
2. **Chuẩn hóa tọa độ** (normalization) theo bounding box cơ thể/bàn tay
3. **Trực quan hóa** skeleton từ dữ liệu đã trích xuất
4. **Phân tích thống kê** dataset (phân bố frames, glosses, signers, ...)

> **Đầu vào:** Video đã crop 224×224 từ notebook 02
> **Đầu ra:** File `.npy` chứa ma trận keypoints `[num_frames, 76, 3]`

---
## 1. Import Thư Viện & Định Nghĩa Landmarks

In [ ]:
import os
import cv2
import numpy as np
import mediapipe as mp
import shutil
from glob import glob
from tqdm import tqdm
from pathlib import Path

# MediaPipe Holistic
mp_holistic = mp.solutions.holistic
mp_drawing = mp.solutions.drawing_utils

# ============================================================
# DANH MỤC 76 ĐIỂM KHỚP (LANDMARKS)
# ============================================================

# 34 điểm Body (33 từ MediaPipe + 1 "neck" tổng hợp)
BODY_LANDMARKS = [
    "nose", "leftEyeInner", "leftEye", "leftEyeOuter",
    "rightEyeInner", "rightEye", "rightEyeOuter",
    "leftEar", "rightEar", "mouthLeft", "mouthRight",
    "leftShoulder", "rightShoulder",
    "leftElbow", "rightElbow", "leftWrist", "rightWrist",
    "leftPinky", "rightPinky", "leftIndex", "rightIndex",
    "leftThumb", "rightThumb",
    "leftHip", "rightHip", "leftKnee", "rightKnee",
    "leftAnkle", "rightAnkle",
    "leftHeel", "rightHeel", "leftFootIndex", "rightFootIndex",
    "neck"  # Điểm tổng hợp: trung bình 2 vai
]

# 21 điểm cho mỗi bàn tay
HAND_LANDMARKS = [
    "wrist", "indexTip", "indexDIP", "indexPIP", "indexMCP",
    "middleTip", "middleDIP", "middlePIP", "middleMCP",
    "ringTip", "ringDIP", "ringPIP", "ringMCP",
    "littleTip", "littleDIP", "littlePIP", "littleMCP",
    "thumbTip", "thumbIP", "thumbMP", "thumbCMC",
]

# 42 điểm Hands = 21 × 2 (suffix _0 = tay trái, _1 = tay phải)
HANDS_LANDMARKS = [
    id + suffix 
    for id in HAND_LANDMARKS 
    for suffix in ["_0", "_1"]
]

# Tổng: 34 + 42 = 76 điểm
LANDMARKS = BODY_LANDMARKS + HANDS_LANDMARKS

print(f"Đã cấu hình danh mục! Tổng: {len(LANDMARKS)} điểm khớp "
      f"({len(BODY_LANDMARKS)} Body + {len(HANDS_LANDMARKS)} Hands).")

---
## 2. Chuẩn Hóa Tọa Độ (Normalization)

### Body Normalization
- Tính bounding box từ 6 anchor landmarks: `nose`, `leftShoulder`,
  `rightShoulder`, `leftHip`, `rightHip`, `neck`
- Scale factor: `1.6×` để bao phủ cả vùng xung quanh
- Chuyển tọa độ về khoảng `[-0.5, 0.5]`

### Hand Normalization
- Mỗi bàn tay được chuẩn hóa **độc lập** theo bounding box riêng
- Kết quả cũng ở khoảng `[-0.5, 0.5]`

In [ ]:
class SingleBodyDictNormalize:
    """
    Chuẩn hóa tọa độ body landmarks theo bounding box 
    của 6 anchor points (nose, shoulders, hips, neck).
    """
    ANCHOR_LANDMARKS = [
        "nose", "leftShoulder", "rightShoulder", 
        "leftHip", "rightHip", "neck"
    ]
    
    def __call__(self, row: dict) -> dict:
        sequence_size = len(row["leftEar"])
        
        for i in range(sequence_size):
            x_coords = [
                row[name][i][0] for name in self.ANCHOR_LANDMARKS 
                if row[name][i][0] != 0
            ]
            y_coords = [
                row[name][i][1] for name in self.ANCHOR_LANDMARKS 
                if row[name][i][1] != 0
            ]
            
            if not x_coords or not y_coords:
                continue
            
            min_x, max_x = min(x_coords), max(x_coords)
            min_y, max_y = min(y_coords), max(y_coords)
            
            # Scale factor 1.6× để bao phủ vùng xung quanh
            dx = (max_x - min_x) * 1.6
            dy = (max_y - min_y) * 1.6
            if dx <= 0 or dy <= 0:
                continue
            
            center_x = (max_x + min_x) / 2
            center_y = (max_y + min_y) / 2
            
            box_min_x = center_x - dx / 2
            box_min_y = center_y - dy / 2
            
            for key in BODY_LANDMARKS:
                x, y, z = row[key][i]
                if x == 0 and y == 0:
                    continue
                row[key][i] = (
                    (x - box_min_x) / dx - 0.5,
                    (y - box_min_y) / dy - 0.5,
                    z
                )
        return row


class SingleHandDictNormalize:
    """
    Chuẩn hóa tọa độ hand landmarks theo bounding box
    riêng của từng bàn tay.
    """
    def __call__(self, row: dict) -> dict:
        sequence_size = len(row["leftEar"])
        
        for suffix in ["_0", "_1"]:
            for i in range(sequence_size):
                x_coords = [
                    row[name + suffix][i][0] for name in HAND_LANDMARKS 
                    if row[name + suffix][i][0] != 0
                ]
                y_coords = [
                    row[name + suffix][i][1] for name in HAND_LANDMARKS 
                    if row[name + suffix][i][1] != 0
                ]
                
                if not x_coords or not y_coords:
                    continue
                
                min_x, max_x = min(x_coords), max(x_coords)
                min_y, max_y = min(y_coords), max(y_coords)
                dx, dy = max_x - min_x, max_y - min_y
                
                if dx <= 0 or dy <= 0:
                    continue
                
                for name in HAND_LANDMARKS:
                    x, y, z = row[name + suffix][i]
                    if x == 0 and y == 0:
                        continue
                    row[name + suffix][i] = (
                        (x - min_x) / dx - 0.5,
                        (y - min_y) / dy - 0.5,
                        z
                    )
        return row


# Khởi tạo normalizers
body_norm = SingleBodyDictNormalize()
hand_norm = SingleHandDictNormalize()

print("Đã khởi tạo Body và Hand normalizers.")

---
## 3. Pipeline Trích Xuất Keypoints 3D

Hàm chính `extract_sign_language_features()` thực hiện:
1. Đọc video frame-by-frame
2. Chạy MediaPipe Holistic trên mỗi frame
3. Trích xuất 34 body landmarks + 42 hand landmarks
4. Tính điểm "neck" tổng hợp (trung bình 2 vai)
5. Chuẩn hóa tọa độ (body + hand)
6. Lưu kết quả dưới dạng file `.npy`

> **Tùy chọn:** Bật `visualize=True` để tạo video overlay skeleton.

In [ ]:
# Khởi tạo MediaPipe Holistic (dùng chung cho cả pipeline)
holistic = mp_holistic.Holistic(static_image_mode=False, model_complexity=1)

# Mapping tên landmark → index trong MediaPipe
POSE_MAP = {
    "nose": 0, "leftEyeInner": 1, "leftEye": 2, "leftEyeOuter": 3,
    "rightEyeInner": 4, "rightEye": 5, "rightEyeOuter": 6,
    "leftEar": 7, "rightEar": 8, "mouthLeft": 9, "mouthRight": 10,
    "leftShoulder": 11, "rightShoulder": 12,
    "leftElbow": 13, "rightElbow": 14, "leftWrist": 15, "rightWrist": 16,
    "leftPinky": 17, "rightPinky": 18, "leftIndex": 19, "rightIndex": 20,
    "leftThumb": 21, "rightThumb": 22,
    "leftHip": 23, "rightHip": 24,
    "leftKnee": 25, "rightKnee": 26, "leftAnkle": 27, "rightAnkle": 28,
    "leftHeel": 29, "rightHeel": 30, "leftFootIndex": 31, "rightFootIndex": 32
}

HAND_MAP = {
    "wrist": 0, "thumbCMC": 1, "thumbMP": 2, "thumbIP": 3, "thumbTip": 4,
    "indexMCP": 5, "indexPIP": 6, "indexDIP": 7, "indexTip": 8,
    "middleMCP": 9, "middlePIP": 10, "middleDIP": 11, "middleTip": 12,
    "ringMCP": 13, "ringPIP": 14, "ringDIP": 15, "ringTip": 16,
    "littleMCP": 17, "littlePIP": 18, "littleDIP": 19, "littleTip": 20
}


def extract_sign_language_features(video_path, npy_out_path, 
                                    visualize=False, vis_out_path=None):
    """
    Trích xuất 76 keypoints 3D từ video ngôn ngữ ký hiệu.
    
    Parameters
    ----------
    video_path : str
        Đường dẫn đến video đầu vào.
    npy_out_path : str
        Đường dẫn lưu file .npy đầu ra.
    visualize : bool
        Nếu True, tạo video overlay skeleton.
    vis_out_path : str, optional
        Đường dẫn lưu video overlay.
    
    Returns
    -------
    bool
        True nếu trích xuất thành công, False nếu thất bại.
    """
    cap = cv2.VideoCapture(video_path)
    if not cap.isOpened():
        return False
    
    fps = cap.get(cv2.CAP_PROP_FPS)
    orig_w = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
    orig_h = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
    
    # Tạo video writer cho visualization (nếu cần)
    video_writer = None
    if visualize and vis_out_path:
        pane_w = 400
        pane_h = int(pane_w * (orig_h / orig_w))
        os.makedirs(os.path.dirname(vis_out_path), exist_ok=True)
        video_writer = cv2.VideoWriter(
            vis_out_path, cv2.VideoWriter_fourcc(*"mp4v"), 
            fps, (pane_w * 3, pane_h)
        )
        
    # Dictionary lưu sequence cho từng landmark
    video_sequence_dict = {name: [] for name in LANDMARKS}
    
    while True:
        ret, frame = cap.read()
        if not ret:
            break
        
        rgb_frame = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
        results = holistic.process(rgb_frame)
        
        # ===== VISUALIZATION (tùy chọn) =====
        if visualize and video_writer:
            pane_w_curr = 400
            pane_h_curr = int(pane_w_curr * (orig_h / orig_w))
            pane_original = cv2.resize(frame, (pane_w_curr, pane_h_curr))
            frame_skeleton_only = np.zeros_like(frame)
            frame_overlay = frame.copy()
            
            def draw_mp_skeleton(img):
                if results.pose_landmarks:
                    mp_drawing.draw_landmarks(
                        img, results.pose_landmarks, 
                        mp_holistic.POSE_CONNECTIONS
                    )
                if results.left_hand_landmarks:
                    mp_drawing.draw_landmarks(
                        img, results.left_hand_landmarks, 
                        mp_holistic.HAND_CONNECTIONS
                    )
                if results.right_hand_landmarks:
                    mp_drawing.draw_landmarks(
                        img, results.right_hand_landmarks, 
                        mp_holistic.HAND_CONNECTIONS
                    )
            
            draw_mp_skeleton(frame_skeleton_only)
            draw_mp_skeleton(frame_overlay)
            
            pane_skeleton = cv2.resize(frame_skeleton_only, (pane_w_curr, pane_h_curr))
            pane_overlay = cv2.resize(frame_overlay, (pane_w_curr, pane_h_curr))
            combined_pane = np.hstack((pane_original, pane_skeleton, pane_overlay))
            video_writer.write(combined_pane)
        
        # ===== TRÍCH XUẤT BODY LANDMARKS =====
        pose_data = {name: (0.0, 0.0, 0.0) for name in BODY_LANDMARKS}
        
        if results.pose_landmarks:
            for name, idx in POSE_MAP.items():
                lm = results.pose_landmarks.landmark[idx]
                pose_data[name] = (lm.x, lm.y, lm.z)
            
            # Tính điểm "neck" = trung bình 2 vai
            ls = pose_data["leftShoulder"]
            rs = pose_data["rightShoulder"]
            pose_data["neck"] = (
                (ls[0] + rs[0]) / 2, 
                (ls[1] + rs[1]) / 2, 
                (ls[2] + rs[2]) / 2
            )
        
        for name in BODY_LANDMARKS:
            video_sequence_dict[name].append(pose_data[name])
        
        # ===== TRÍCH XUẤT HAND LANDMARKS =====
        for suffix, hand_landmarks in [
            ("_0", results.left_hand_landmarks), 
            ("_1", results.right_hand_landmarks)
        ]:
            hand_data = {
                name + suffix: (0.0, 0.0, 0.0) for name in HAND_LANDMARKS
            }
            if hand_landmarks:
                for name, idx in HAND_MAP.items():
                    lm = hand_landmarks.landmark[idx]
                    hand_data[name + suffix] = (lm.x, lm.y, lm.z)
            
            for name in HAND_LANDMARKS:
                video_sequence_dict[name + suffix].append(
                    hand_data[name + suffix]
                )
    
    cap.release()
    if video_writer:
        video_writer.release()
    
    # ===== CHUẨN HÓA TỌA ĐỘ =====
    video_sequence_dict = body_norm(video_sequence_dict)
    video_sequence_dict = hand_norm(video_sequence_dict)
    
    # ===== CHUYỂN ĐỔI THÀNH NUMPY ARRAY =====
    seq_len = len(video_sequence_dict["neck"])
    if seq_len == 0:
        return False
    
    frames_list = []
    for i in range(seq_len):
        frame_features = [video_sequence_dict[name][i] for name in LANDMARKS]
        frames_list.append(frame_features)
        
    np_array_data = np.array(frames_list, dtype=np.float32)
    
    # Shape: [num_frames, 76, 3]
    os.makedirs(os.path.dirname(npy_out_path), exist_ok=True)
    np.save(npy_out_path, np_array_data)
    
    return True

print(" Pipeline core trích xuất 3D keypoints đã sẵn sàng!")

---
## 4. Cấu Hình & Chạy Trích Xuất Keypoints

Dataset VSL-400 có 400 class × ~60 video/class ≈ 24,000+ video.
Để tránh quá tải trên Kaggle, chia thành **4 batch**, mỗi batch 100 classes.

> 💡 **Tip:** Thay đổi `CURRENT_BATCH` (1→4) để chạy từng batch.

In [ ]:
# ============================================================
# CẤU HÌNH — Thay đổi đường dẫn phù hợp với máy của bạn
# ============================================================
CURRENT_BATCH = 1  # Nhập giá trị: 1, 2, 3, hoặc 4

INPUT_DIR = "data_splited/train"          # Thư mục video đã chia
OUTPUT_KEYPOINTS_DIR = "keypoints"         # Thư mục lưu file .npy

# Dọn dẹp dữ liệu cũ (nếu cần)
if os.path.exists(OUTPUT_KEYPOINTS_DIR):
    print(f" Thư mục {OUTPUT_KEYPOINTS_DIR} đã tồn tại.")
    # shutil.rmtree(OUTPUT_KEYPOINTS_DIR)
    # print("Đã dọn sạch dữ liệu cũ.")

# Tìm tất cả class
all_classes = sorted([
    f for f in os.listdir(INPUT_DIR) 
    if os.path.isdir(os.path.join(INPUT_DIR, f))
])
total_classes = len(all_classes)

# Chia thành 4 batch
chunks = np.array_split(all_classes, 4)
active_classes = list(chunks[CURRENT_BATCH - 1])

print(f" Dataset gốc: {total_classes} classes.")
print(f" TIẾN TRÌNH: BATCH SỐ {CURRENT_BATCH}/4")
print(f" Số lượng class phụ trách: {len(active_classes)} "
      f"(Từ '{active_classes[0]}' đến '{active_classes[-1]}')")

### Chạy trích xuất keypoints

In [ ]:
for class_name in active_classes:
    class_input_path = os.path.join(INPUT_DIR, class_name)
    video_files = glob(os.path.join(class_input_path, "*.mp4"))
    
    print(f"\n Trích xuất: '{class_name}' ({len(video_files)} videos)")
    
    for video_path in tqdm(video_files, desc=f"Batch {CURRENT_BATCH} → {class_name}"):
        video_id = Path(video_path).stem
        npy_out_path = os.path.join(
            OUTPUT_KEYPOINTS_DIR, class_name, f"{video_id}.npy"
        )
        
        # Bỏ qua nếu đã xử lý
        if os.path.exists(npy_out_path):
            continue
            
        extract_sign_language_features(video_path, npy_out_path, visualize=False)

print(f"\n🎉 HOÀN THÀNH XỬ LÝ TOÀN BỘ BATCH SỐ {CURRENT_BATCH}/4!")

---
## 5. Trực Quan Hóa Skeleton (Visualization)

Tạo video animation từ dữ liệu keypoints đã trích xuất, gồm 3 panel:
1. **Full Body** — Skeleton toàn thân (xanh lá)
2. **Left Hand** — Skeleton bàn tay trái (cam)
3. **Right Hand** — Skeleton bàn tay phải (xanh dương)

In [ ]:
# Kết nối các điểm trên cơ thể
BODY_CONNECTIONS = [
    ("nose", "neck"), ("neck", "rightShoulder"), ("neck", "leftShoulder"),
    ("rightShoulder", "rightElbow"), ("rightElbow", "rightWrist"),
    ("leftShoulder", "leftElbow"), ("leftElbow", "leftWrist"),
    ("rightShoulder", "leftShoulder"),
    ("leftShoulder", "leftHip"), ("rightShoulder", "rightHip"),
    ("leftHip", "rightHip"),
    ("leftHip", "leftKnee"), ("leftKnee", "leftAnkle"),
    ("rightHip", "rightKnee"), ("rightKnee", "rightAnkle")
]

# Kết nối các ngón tay
FINGER_CHAINS = [
    ["wrist", "thumbCMC", "thumbMP", "thumbIP", "thumbTip"],
    ["wrist", "indexMCP", "indexPIP", "indexDIP", "indexTip"],
    ["wrist", "middleMCP", "middlePIP", "middleDIP", "middleTip"],
    ["wrist", "ringMCP", "ringPIP", "ringDIP", "ringTip"],
    ["wrist", "littleMCP", "littlePIP", "littleDIP", "littleTip"]
]


def create_normalized_motion_video(npy_path, out_video_path, fps=25):
    """
    Tạo video animation từ dữ liệu keypoints .npy.
    
    Parameters
    ----------
    npy_path : str
        Đường dẫn file .npy chứa keypoints.
    out_video_path : str
        Đường dẫn lưu video animation.
    fps : int
        Frame rate đầu ra.
    """
    # Tạo mapping landmark → index
    LM_IDX = {name: i for i, name in enumerate(LANDMARKS)}
    
    data = np.load(npy_path)
    total_frames = len(data)
    S = 300  # Kích thước mỗi panel
    
    os.makedirs(os.path.dirname(out_video_path), exist_ok=True)
    out_writer = cv2.VideoWriter(
        out_video_path, cv2.VideoWriter_fourcc(*"mp4v"), 
        fps, (S * 3, S)
    )
    
    def to_pixels(name, kp_frame):
        """Chuyển tọa độ chuẩn hóa thành pixel."""
        if name not in LM_IDX:
            return None
        x, y, z = kp_frame[LM_IDX[name]]
        if abs(x - 0.5) < 1e-4 and abs(y - 0.5) < 1e-4:
            return None
        return (int(x * (S - 30)) + 15, int(y * (S - 30)) + 15)
    
    for t in range(total_frames):
        kp_frame = data[t] + 0.5  # Shift từ [-0.5, 0.5] → [0, 1]
        
        img_body = np.zeros((S, S, 3), dtype=np.uint8)
        img_left_hand = np.zeros((S, S, 3), dtype=np.uint8)
        img_right_hand = np.zeros((S, S, 3), dtype=np.uint8)
        
        # Vẽ body skeleton
        for a, b in BODY_CONNECTIONS:
            pa = to_pixels(a, kp_frame)
            pb = to_pixels(b, kp_frame)
            if pa and pb:
                cv2.line(img_body, pa, pb, (0, 255, 0), 2)
                cv2.circle(img_body, pa, 4, (0, 0, 255), -1)
        
        # Vẽ hand skeleton
        for suffix, canvas, color in [
            ("_0", img_left_hand, (255, 165, 0)),
            ("_1", img_right_hand, (0, 165, 255))
        ]:
            for chain in FINGER_CHAINS:
                for i in range(len(chain) - 1):
                    pa = to_pixels(chain[i] + suffix, kp_frame)
                    pb = to_pixels(chain[i+1] + suffix, kp_frame)
                    if pa and pb:
                        cv2.line(canvas, pa, pb, color, 2)
                        cv2.circle(canvas, pa, 3, (255, 255, 255), -1)
        
        # Thêm tiêu đề
        cv2.putText(img_body, "1. Full Body Norm (3D)", 
                    (10, 25), cv2.FONT_HERSHEY_SIMPLEX, 0.5, (255, 255, 255), 2)
        cv2.putText(img_left_hand, "2. Left Hand Norm", 
                    (10, 25), cv2.FONT_HERSHEY_SIMPLEX, 0.5, (255, 165, 0), 2)
        cv2.putText(img_right_hand, "3. Right Hand Norm", 
                    (10, 25), cv2.FONT_HERSHEY_SIMPLEX, 0.5, (0, 165, 255), 2)
        
        combined_frame = np.hstack((img_body, img_left_hand, img_right_hand))
        out_writer.write(combined_frame)
    
    out_writer.release()
    print(f"Đã tạo video animation: {out_video_path}")

### Test trực quan hóa trên 10 video mẫu

In [ ]:
# ============================================================
# CẤU HÌNH — Thay đổi tùy theo dữ liệu của bạn
# ============================================================
TEST_VIS_DIR = "test_vis"

if 'active_classes' in locals() and len(active_classes) > 0:
    test_class = active_classes[0]
    test_class_input = os.path.join(INPUT_DIR, test_class)
    test_videos = sorted(glob(os.path.join(test_class_input, "*.mp4")))
    
    if test_videos:
        videos_to_test = test_videos[:10]
        print(f"Test trực quan hóa: {len(videos_to_test)} video "
              f"của lớp '{test_class}'")
        os.makedirs(TEST_VIS_DIR, exist_ok=True)
        
        for idx, video_test_path in enumerate(videos_to_test):
            video_test_id = Path(video_test_path).stem
            npy_test_out = os.path.join(
                OUTPUT_KEYPOINTS_DIR, test_class, f"{video_test_id}.npy"
            )
            video_vis_out = os.path.join(
                TEST_VIS_DIR, f"{video_test_id}_overlay.mp4"
            )
            
            print(f"   [{idx+1}/10] Đang xử lý: "
                  f"{Path(video_test_path).name} ...", end="")
            
            # Tạo lại nếu chưa có
            if not os.path.exists(npy_test_out):
                success = extract_sign_language_features(
                    video_test_path, npy_test_out,
                    visualize=True, vis_out_path=video_vis_out
                )
                if success:
                    print("  Xong!")
                else:
                    print("  Thất bại!")
            else:
                print(" ⏩ Đã có sẵn.")
        
        # Tạo video animation cho các file .npy đã trích xuất
        print("\n Tạo video animation từ keypoints đã chuẩn hóa...")
        npy_files = sorted(glob(os.path.join(
            OUTPUT_KEYPOINTS_DIR, test_class, "*.npy"
        )))[:10]
        
        for npy_path in npy_files:
            v_id = Path(npy_path).stem
            norm_motion_path = os.path.join(
                TEST_VIS_DIR, f"{v_id}_normalized_motion.mp4"
            )
            create_normalized_motion_video(npy_path, norm_motion_path, fps=25)
        
        print("\n🎉 Đã hoàn thành trực quan hóa!")
    else:
        print(f"Không tìm thấy video trong '{test_class_input}'")
else:
    print("Chưa có active_classes. Hãy chạy cell cấu hình ở mục 4.")

---
## 6. Phân Tích Thống Kê Dataset (EDA)

Phân tích tổng quan dữ liệu đã trích xuất:
- Phân bố số lượng frames
- Phân bố số video theo gloss
- Thống kê shape dữ liệu

In [ ]:
def analyze_keypoints_dataset(keypoints_dir):
    """
    Phân tích thống kê dataset keypoints đã trích xuất.
    
    Parameters
    ----------
    keypoints_dir : str
        Thư mục chứa các file .npy đã trích xuất.
    """
    if not os.path.exists(keypoints_dir):
        print(f" Thư mục không tồn tại: {keypoints_dir}")
        return
    
    all_npy_files = glob(os.path.join(keypoints_dir, "*", "*.npy"))
    
    if not all_npy_files:
        print(f"Không tìm thấy file .npy nào trong {keypoints_dir}")
        return
    
    print(f"Phân tích dataset: {keypoints_dir}")
    print(f"   Tổng file .npy: {len(all_npy_files)}")
    
    # Thu thập thống kê
    frame_counts = []
    shapes = []
    class_counts = {}
    errors = 0
    
    for npy_path in tqdm(all_npy_files, desc="Đang phân tích"):
        try:
            data = np.load(npy_path)
            frame_counts.append(data.shape[0])
            shapes.append(data.shape)
            
            class_name = os.path.basename(os.path.dirname(npy_path))
            class_counts[class_name] = class_counts.get(class_name, 0) + 1
        except Exception:
            errors += 1
    
    frame_counts = np.array(frame_counts)
    
    print(f"\n{'='*50}")
    print(f" THỐNG KÊ TỔNG QUAN")
    print(f"{'='*50}")
    print(f"  Tổng video đã trích xuất: {len(frame_counts)}")
    print(f"  Số class (gloss):         {len(class_counts)}")
    print(f"  File lỗi:                 {errors}")
    
    print(f"\n PHÂN BỐ SỐ FRAMES:")
    print(f"  Min:    {frame_counts.min()} frames")
    print(f"  Max:    {frame_counts.max()} frames")
    print(f"  Mean:   {frame_counts.mean():.1f} frames")
    print(f"  Median: {np.median(frame_counts):.1f} frames")
    print(f"  Std:    {frame_counts.std():.1f} frames")
    
    print(f"\n PHÂN BỐ SỐ VIDEO THEO CLASS:")
    video_per_class = np.array(list(class_counts.values()))
    print(f"  Min:    {video_per_class.min()} videos/class")
    print(f"  Max:    {video_per_class.max()} videos/class")
    print(f"  Mean:   {video_per_class.mean():.1f} videos/class")
    
    # Shape nhất quán?
    unique_shapes_last2 = set(s[1:] for s in shapes)
    if len(unique_shapes_last2) == 1:
        print(f"\n Shape nhất quán: [num_frames, {shapes[0][1]}, {shapes[0][2]}]")
    else:
        print(f"\n Shape KHÔNG nhất quán: {unique_shapes_last2}")
    
    return frame_counts, class_counts


# Chạy phân tích (nếu đã có dữ liệu)
if os.path.exists(OUTPUT_KEYPOINTS_DIR):
    frame_counts, class_counts = analyze_keypoints_dataset(OUTPUT_KEYPOINTS_DIR)
else:
    print(f"Chưa có dữ liệu tại '{OUTPUT_KEYPOINTS_DIR}'. "
          "Hãy chạy trích xuất keypoints trước.")

---
## 7. Tổng Kết Pipeline

### Cấu trúc dữ liệu đầu ra:

```
keypoints/
├── Anh/
│   ├── 000861.npy     # shape: [num_frames, 76, 3]
│   ├── 000862.npy
│   └── ...
├── Chị/
│   ├── 000002.npy
│   └── ...
└── ...
```

### Định dạng mỗi file `.npy`:
| Dimension | Ý nghĩa |
|-----------|---------|
| axis 0 | Số frames (biến thiên theo video) |
| axis 1 | 76 landmarks (34 body + 42 hands) |
| axis 2 | 3 tọa độ (x, y, z) đã chuẩn hóa |

### Tổng kết pipeline hoàn chỉnh:

```
┌───────────────────────────────────────────────┐
│  01_data_collection.ipynb                        │
│  Thu thập & tổ chức dữ liệu thô              │
│  Đầu ra: Metadata JSON + Video phân loại      │
└───────────────────┬───────────────────────────┘
                    │
                    ▼
┌───────────────────────────────────────────────┐
│  02_data_cleaning_and_imputation.ipynb           │
│  Tiền xử lý video (TBL + Crop 224×224)        │
│  Đầu ra: Video chuẩn hóa 224×224              │
└───────────────────┬───────────────────────────┘
                    │
                    ▼
┌───────────────────────────────────────────────┐
│  03_exploratory_data_analysis.ipynb              │
│  Trích xuất 76 keypoints 3D + EDA             │
│  Đầu ra: File .npy [frames, 76, 3]            │
└───────────────────────────────────────────────┘
```